# VQE-style prime identification

For each integer n, optimize a 2-qubit variational circuit to
minimize an energy function E(n) = ⟨Z₀Z₁⟩ × (0.5 + factor_count(n)).
Numbers with few factors (primes) naturally land at lower energies.

In [ ]:
import pennylane as qml
import numpy as np

## Setup

In [ ]:
dev = qml.device("default.qubit", wires=2)

def count_small_factors(n):
    if n <= 1:
        return 0
    count = 0
    d = 2
    while d * d <= n:
        while n % d == 0:
            count += 1
            n //= d
        d += 1
    if n > 1:
        count += 1
    return count

## Variational circuit

In [ ]:
@qml.qnode(dev, diff_method="parameter-shift")
def energy_circuit(params):
    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.Z(0) @ qml.Z(1))

def prime_energy(n, params):
    return energy_circuit(params) * (0.5 + count_small_factors(n))

print(qml.draw(energy_circuit)(np.zeros(2)))

## Optimise for each n

In [ ]:
rng = np.random.default_rng(42)
test_values = [2, 3, 5, 7, 11, 4, 6, 8, 9, 15]

results = []
for n in test_values:
    params = rng.normal(0, 0.5, size=2).astype(float, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=0.1)
    for _ in range(30):
        params = opt.step(lambda p, n=n: prime_energy(n, p), params)
    e = float(prime_energy(n, params))
    results.append((n, e))

results.sort(key=lambda t: t[1])
print(f"{'n':>4s}  {'energy':>8s}  {'prime?':>6s}")
print("-" * 25)
for n, e in results:
    is_prime = all(n % d != 0 for d in range(2, int(n**0.5) + 1)) and n >= 2
    print(f"{n:4d}  {e:+8.4f}  {'yes' if is_prime else 'no':>6s}")